# Persona Generators: Step-by-Step Pipeline
## Reproducing "Generating Diverse Synthetic Personas at Scale" (Paglieri et al., 2026)

This notebook walks through the complete pipeline with **intermediate saves** at every step:

| Step | What happens | Saved artifact |
|------|-------------|----------------|
| 0 | Setup & configuration | — |
| 1 | Questionnaire generation | `outputs/01_questionnaire.json` |
| 2a | Sobol diversity sampling | `outputs/02a_diversity_positions.json` |
| 2b | Stage 1: Autoregressive descriptors | `outputs/02b_stage1_descriptors.json` |
| 2c | Stage 2: Full persona expansion | `outputs/02c_stage2_personas.json` |
| 3 | Concordia evaluation (Logic of Appropriateness) | `outputs/03_evaluation_results.json` |
| 4 | Diversity metrics | `outputs/04_diversity_metrics.json` |

Each artifact is saved as JSON for inspection. You can stop at any step, examine the outputs, and resume.

---
## Step 0: Setup & Configuration

In [1]:
import os
import sys
import json
import time
import numpy as np
from pathlib import Path

# Create outputs directory for all intermediate artifacts
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

def save_artifact(data, filename, step_name=""):
    """Save a pipeline artifact to JSON for inspection."""
    path = OUTPUT_DIR / filename
    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)
    size_kb = path.stat().st_size / 1024
    print(f"  ✅ Saved: {path} ({size_kb:.1f} KB)")
    return path

def load_artifact(filename):
    """Load a previously saved artifact."""
    path = OUTPUT_DIR / filename
    with open(path) as f:
        return json.load(f)

print("Output directory:", OUTPUT_DIR.resolve())

Output directory: C:\Users\G25971483\Desktop\Projects\recover_entropy_collapse\Persona_Generator\outputs


In [2]:
# ── Configuration ──
# Load OpenRouter API key from .env
from dotenv import load_dotenv
load_dotenv()
import os
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

# Import all pipeline modules
from config import *
from llm_client import call_llm, call_llm_json
from questionnaire_generator import generate_questionnaire, print_questionnaire, Question, Questionnaire
from diversity_sampler import generate_diversity_positions, positions_to_labels, print_positions
from persona_generator import (
    generate_stage1_descriptors, expand_persona_stage2,
    generate_personas, Persona, print_personas,
)
from concordia_evaluator import evaluate_population, print_evaluation_results
from diversity_metrics import compute_all_metrics, print_metrics, normalize_embeddings

# ── Pipeline parameters ──
SHORT_DESCRIPTION = "elderly rural japan 2010"
NUM_PERSONAS = 25
PERSONA_FORMAT = "first_person"  # Options: "first_person", "logic_of_appropriateness", "rule_based"
SEED = 42

print(f"Context: {SHORT_DESCRIPTION}")
print(f"Population size: {NUM_PERSONAS}")
print(f"Persona format: {PERSONA_FORMAT}")
print(f"Seed: {SEED}")
print(f"Questionnaire model: {QUESTIONNAIRE_MODEL}")
print(f"Persona model: {PERSONA_MODEL}")
print(f"Simulation model: {SIMULATION_MODEL}")
print(f"API key set: {'Yes' if OPENROUTER_API_KEY else 'NO — set it above!'}")

Context: elderly rural japan 2010
Population size: 25
Persona format: first_person
Seed: 42
Questionnaire model: meta-llama/llama-3.1-405b-instruct
Persona model: meta-llama/llama-3.1-405b-instruct
Simulation model: meta-llama/llama-3.1-405b-instruct
API key set: Yes


---
## Step 1: Questionnaire Generation

**Paper reference:** Section 3.2 and Appendix A

Takes a short description and produces:
- **Context `c`**: Expanded scenario description
- **Dimensions `D`**: 2-3 diversity axes
- **Items `I`**: Likert-scale survey questions (≥3 per axis, with reverse-coded items)

The generator uses few-shot prompting with real psychometric questionnaires (Big Five, DASS, SVO, NFCS) as examples.

> **Key design choice:** The Persona Generator (Step 2) will see `c` and `D` but **never** the items `I`. The items are held back for evaluation only. This prevents the generator from gaming the evaluation.

In [3]:
print("=" * 70)
print("STEP 1: QUESTIONNAIRE GENERATION")
print("=" * 70)
print(f"\nInput: \"{SHORT_DESCRIPTION}\"")
print("Calling LLM to expand into full questionnaire...\n")

questionnaire = generate_questionnaire(SHORT_DESCRIPTION)
print_questionnaire(questionnaire)

STEP 1: QUESTIONNAIRE GENERATION

Input: "elderly rural japan 2010"
Calling LLM to expand into full questionnaire...


QUESTIONNAIRE: elderly rural japan 2010

Context:
  A survey of elderly residents in a rural Japanese village in 2010, exploring their feelings about community, technology adoption, and traditional values.

Dimensions: ['community_cohesion', 'technology_adoption', 'adherence_to_tradition']

Questions (9 total):

  Q1 [community_cohesion]
  Preprompt: An interviewer asks {player_name} how much they agree or disagree with the following statement:
  Statement: {player_name} feels a strong sense of belonging to the village community.

  Q2 [community_cohesion]
  Preprompt: An interviewer asks {player_name} how much they agree or disagree with the following statement:
  Statement: {player_name} believes that most people in this village can be trusted.

  Q3 [community_cohesion] [REVERSE-CODED]
  Preprompt: An interviewer asks {player_name} how much they agree or disagree wi

In [4]:
# ── Save questionnaire artifact ──
questionnaire_artifact = {
    "short_description": questionnaire.short_description,
    "context": questionnaire.context,
    "dimensions": questionnaire.dimensions,
    "num_dimensions": len(questionnaire.dimensions),
    "num_questions": len(questionnaire.questions),
    "questions_per_dimension": {
        dim: len(questionnaire.get_questions_for_dimension(dim))
        for dim in questionnaire.dimensions
    },
    "questions": [
        {
            "preprompt": q.preprompt,
            "statement": q.statement,
            "dimension": q.dimension,
            "ascending_scale": q.ascending_scale,
            "choices": q.choices,
        }
        for q in questionnaire.questions
    ],
}

save_artifact(questionnaire_artifact, "01_questionnaire.json")

  ✅ Saved: outputs\01_questionnaire.json (4.8 KB)


WindowsPath('outputs/01_questionnaire.json')

In [7]:
questionnaire_artifact = load_artifact("01_questionnaire.json")
questionnaire = Questionnaire(
    short_description=questionnaire_artifact["short_description"],
    context=questionnaire_artifact["context"],
    dimensions=questionnaire_artifact["dimensions"],
    questions=[
        Question(
            preprompt=q["preprompt"],
            statement=q["statement"],
            choices=q["choices"],              # or LIKERT_SCALE from config if you prefer
            ascending_scale=q["ascending_scale"],
            dimension=q["dimension"],
        )
        for q in questionnaire_artifact["questions"]
    ],
)

### 📋 Inspect: What the questionnaire looks like

The generated questionnaire contains:
- A **context** describing the scenario
- **Diversity axes** (e.g., community cohesion, technology adoption, tradition)
- **Survey items** grouped by axis, each with a Likert scale

Some items are **reverse-coded** (ascending_scale=false), meaning "Strongly agree" indicates a *low* score on that dimension. This is standard psychometric practice to catch acquiescence bias.

In [8]:
# Quick inspection of the questionnaire
print(f"Context: {questionnaire.context}\n")
print(f"Diversity Axes ({len(questionnaire.dimensions)}):")
for dim in questionnaire.dimensions:
    qs = questionnaire.get_questions_for_dimension(dim)
    rev_count = sum(1 for q in qs if not q.ascending_scale)
    print(f"  • {dim}: {len(qs)} items ({rev_count} reverse-coded)")

print(f"\nSample questions:")
for q in questionnaire.questions[:3]:
    rev = " [REVERSE]" if not q.ascending_scale else ""
    print(f"  [{q.dimension}]{rev}: \"{q.statement[:80]}...\"")

Context: A survey of elderly residents in a rural Japanese village in 2010, exploring their feelings about community, technology adoption, and traditional values.

Diversity Axes (3):
  • community_cohesion: 3 items (1 reverse-coded)
  • technology_adoption: 3 items (1 reverse-coded)
  • adherence_to_tradition: 3 items (1 reverse-coded)

Sample questions:
  [community_cohesion]: "{player_name} feels a strong sense of belonging to the village community...."
  [community_cohesion]: "{player_name} believes that most people in this village can be trusted...."
  [community_cohesion] [REVERSE]: "{player_name} often feels lonely or isolated from others in the village...."


---
## Step 2a: Quasi-Random Diversity Sampling (Sobol Sequence)

**Paper reference:** Section 3.3

Before generating any personas, we first decide **where each persona should sit** in the diversity space using Sobol quasi-random sequences.

This is the sampling strategy that **dominated** the evolutionary search — after the first extinction event (~100 iterations), only generators using quasi-random Monte Carlo sampling survived.

**Why Sobol beats other approaches:**
- **vs. Uniform random:** Sobol has lower discrepancy (fewer gaps, more even coverage)
- **vs. LLM-generated positions:** LLMs suffer from mode collapse, clustering around "typical" positions
- **vs. Grid sampling:** Sobol handles high dimensions better and doesn't require N to be a perfect power

Each persona gets a position in [0, 1]^K where K = number of diversity axes.

In [9]:
print("=" * 70)
print("STEP 2a: SOBOL QUASI-RANDOM SAMPLING")
print("=" * 70)

dimensions = questionnaire.dimensions
K = len(dimensions)
print(f"\nSampling {NUM_PERSONAS} positions in {K}-dimensional diversity space")
print(f"Axes: {dimensions}\n")

# Generate Sobol positions
positions = generate_diversity_positions(NUM_PERSONAS, K, seed=SEED)
axis_labels = positions_to_labels(positions, dimensions)

# Display positions
print_positions(positions, dimensions)

STEP 2a: SOBOL QUASI-RANDOM SAMPLING

Sampling 25 positions in 3-dimensional diversity space
Axes: ['community_cohesion', 'technology_adoption', 'adherence_to_tradition']


Diversity Positions (25 personas, 3 axes):
----------------------------------------------------------------------
  Persona  1: community_cohesion=0.62 (high), technology_adoption=0.21 (very low), adherence_to_tradition=0.31 (low)
  Persona  2: community_cohesion=0.47 (moderate), technology_adoption=0.12 (extremely low), adherence_to_tradition=0.39 (low)
  Persona  3: community_cohesion=0.29 (very low), technology_adoption=0.92 (extremely high), adherence_to_tradition=0.46 (moderate)
  Persona  4: community_cohesion=0.77 (very high), technology_adoption=0.08 (extremely low), adherence_to_tradition=0.23 (very low)
  Persona  5: community_cohesion=0.32 (low), technology_adoption=0.59 (high), adherence_to_tradition=0.19 (very low)
  Persona  6: community_cohesion=0.88 (extremely high), technology_adoption=0.16 (very lo

In [7]:
# ── Save diversity positions artifact ──
positions_artifact = {
    "num_personas": NUM_PERSONAS,
    "num_dimensions": K,
    "dimensions": dimensions,
    "sampling_method": "sobol_quasi_random",
    "seed": SEED,
    "positions": [
        {
            "persona_index": i,
            "raw_values": positions[i].tolist(),
            "labeled": {
                dim: axis_labels[i][dim]
                for dim in dimensions
            },
        }
        for i in range(NUM_PERSONAS)
    ],
    "statistics": {
        dim: {
            "min": float(positions[:, j].min()),
            "max": float(positions[:, j].max()),
            "mean": float(positions[:, j].mean()),
            "std": float(positions[:, j].std()),
        }
        for j, dim in enumerate(dimensions)
    },
}

save_artifact(positions_artifact, "02a_diversity_positions.json")

  ✅ Saved: outputs\02a_diversity_positions.json (13.1 KB)


WindowsPath('outputs/02a_diversity_positions.json')

In [10]:
positions_artifact = load_artifact("02a_diversity_positions.json")
# Core metadata
NUM_PERSONAS = positions_artifact["num_personas"]
K = positions_artifact["num_dimensions"]
dimensions = positions_artifact["dimensions"]
SEED = positions_artifact.get("seed", None)

# Rebuild positions array (shape: [NUM_PERSONAS, K])
positions = np.array([
    p["raw_values"] for p in positions_artifact["positions"]
])

# Rebuild axis_labels (same structure you had before saving)
axis_labels = [
    pos_entry["labeled"]
    for pos_entry in positions_artifact["positions"]
]

### 📋 Inspect: Interesting trait combinations

Sobol sampling discovers **rare corner combinations** that LLMs would never generate on their own. For example, a persona with *extremely low* community cohesion but *extremely high* tradition adherence — someone who is deeply traditional but completely isolated from their community.

In [11]:
# Find and display interesting extreme combinations
print("Notable extreme combinations discovered by Sobol sampling:\n")
for i, persona_axes in enumerate(axis_labels):
    values = [info["value"] for info in persona_axes.values()]
    labels = [info["label"] for info in persona_axes.values()]

    # Find personas with at least one extreme high AND one extreme low
    has_extreme_high = any(v > 0.85 for v in values)
    has_extreme_low = any(v < 0.15 for v in values)

    if has_extreme_high and has_extreme_low:
        print(f"  Persona {i+1}:")
        for dim, info in persona_axes.items():
            marker = " ◄◄" if info["value"] < 0.15 or info["value"] > 0.85 else ""
            print(f"    {dim}: {info['value']:.3f} ({info['label']}){marker}")
        print()

Notable extreme combinations discovered by Sobol sampling:

  Persona 13:
    community_cohesion: 0.734 (very high)
    technology_adoption: 0.033 (extremely low) ◄◄
    adherence_to_tradition: 0.918 (extremely high) ◄◄

  Persona 15:
    community_cohesion: 0.113 (extremely low) ◄◄
    technology_adoption: 0.477 (moderate)
    adherence_to_tradition: 0.964 (extremely high) ◄◄



---
## Step 2b: Stage 1 — Autoregressive High-Level Descriptors

**Paper reference:** Section 3.3 (Stage 1)

The LLM receives:
- The questionnaire **context** `c`
- The **diversity axes** `D`
- The **Sobol-sampled positions** for each persona

It generates a **high-level descriptor** for each persona: a name and 2-3 sentence sketch of who they are, anchored to their target axis positions.

**Key properties:**
- **Autoregressive**: Each batch sees all previously generated personas to avoid duplication
- **Batched**: Personas generated in groups of 5 (reduces API calls while maintaining awareness)
- The generator sees **context + axes only** — never the survey items

In [9]:
print("=" * 70)
print("STEP 2b: STAGE 1 — AUTOREGRESSIVE HIGH-LEVEL DESCRIPTORS")
print("=" * 70)
print(f"\nGenerating high-level descriptors for {NUM_PERSONAS} personas...")
print(f"Context provided: {questionnaire.context}")
print(f"Axes provided: {dimensions}")
print(f"Items provided: NONE (held back for evaluation)\n")

stage1_results = generate_stage1_descriptors(
    context=questionnaire.context,
    dimensions=dimensions,
    num_personas=NUM_PERSONAS,
    batch_size=5,
    seed=SEED,
)

print(f"\n✅ Generated {len(stage1_results)} Stage 1 descriptors")

STEP 2b: STAGE 1 — AUTOREGRESSIVE HIGH-LEVEL DESCRIPTORS

Generating high-level descriptors for 25 personas...
Context provided: A survey of elderly residents in a rural Japanese village in 2010, exploring their feelings about community, technology adoption, and traditional values.
Axes provided: ['community_cohesion', 'technology_adoption', 'adherence_to_tradition']
Items provided: NONE (held back for evaluation)

  Stage 1: Generated batch 1-5 of 25
  Stage 1: Generated batch 6-10 of 25
  Stage 1: Generated batch 11-15 of 25
  Stage 1: Generated batch 16-20 of 25
  Stage 1: Generated batch 21-25 of 25

✅ Generated 25 Stage 1 descriptors


In [10]:
# ── Save Stage 1 artifact ──
stage1_artifact = {
    "num_personas": len(stage1_results),
    "context_provided": questionnaire.context,
    "dimensions_provided": dimensions,
    "items_provided": "NONE — held back for evaluation",
    "batch_size": 5,
    "descriptors": [
        {
            "persona_index": i,
            "name": s1["name"],
            "descriptor": s1["descriptor"],
            "target_axis_positions": s1["axis_positions"],
            "raw_sobol_values": s1["raw_positions"],
        }
        for i, s1 in enumerate(stage1_results)
    ],
}

save_artifact(stage1_artifact, "02b_stage1_descriptors.json")

  ✅ Saved: outputs\02b_stage1_descriptors.json (24.2 KB)


WindowsPath('outputs/02b_stage1_descriptors.json')

In [12]:
stage1_artifact = load_artifact("02b_stage1_descriptors.json")
NUM_PERSONAS = stage1_artifact["num_personas"]
dimensions = stage1_artifact["dimensions_provided"]
context = stage1_artifact["context_provided"]

# Rebuild the list of stage1_results in the original format
stage1_results = [
    {
        "name": d["name"],
        "descriptor": d["descriptor"],
        "axis_positions": d["target_axis_positions"],   # dict per dim with value/label
        "raw_positions": np.array(d["raw_sobol_values"])
    }
    for d in stage1_artifact["descriptors"]
]

# Optional: separate array of raw Sobol positions
positions = np.array([
    d["raw_sobol_values"] for d in stage1_artifact["descriptors"]
])

### 📋 Inspect: Stage 1 output — high-level persona sketches

Each persona now has a **name** and a **short descriptor** that positions them along the diversity axes. These are the "blueprints" that Stage 2 will expand into full personas.

In [13]:
# Display Stage 1 descriptors
print("Stage 1 Descriptors (name + high-level sketch):\n")
for i, s1 in enumerate(stage1_results):
    print(f"{'─'*60}")
    print(f"Persona {i+1}: {s1['name']}")
    print(f"Target positions:")
    for dim, info in s1["axis_positions"].items():
        print(f"  {dim}: {info['value']:.3f} ({info['label']})")
    print(f"Descriptor:")
    print(f"  {s1['descriptor'][:300]}")
    print()
    if i >= 4:
        print(f"  ... ({len(stage1_results) - 5} more personas)")
        break

Stage 1 Descriptors (name + high-level sketch):

────────────────────────────────────────────────────────────
Persona 1: Yumi
Target positions:
  community_cohesion: 0.620 (high)
  technology_adoption: 0.206 (very low)
  adherence_to_tradition: 0.314 (low)
Descriptor:
  Yumi is a community-oriented person with strong ties to her neighbors (community_cohesion: 0.62). Despite her strong sense of community, she's hesitant to adopt new technology, such as cell phones or the internet (technology_adoption: 0.21). While she respects tradition, she doesn't prioritize adher

────────────────────────────────────────────────────────────
Persona 2: Kazuo
Target positions:
  community_cohesion: 0.475 (moderate)
  technology_adoption: 0.115 (extremely low)
  adherence_to_tradition: 0.390 (low)
Descriptor:
  Kazuo is a moderately community-minded individual who values his relationships with those around him, but also enjoys his alone time (community_cohesion: 0.47). He's extremely resistant to new te

---
## Step 2c: Stage 2 — Parallel Full Persona Expansion

**Paper reference:** Section 3.3 (Stage 2) and Section 5.1 (best evolved solutions)

Each Stage 1 descriptor is independently expanded into a **complete persona description**. This stage runs in parallel (no dependencies between personas).

The paper's evolutionary search discovered three winning formats:

| Format | Best at | Description |
|--------|---------|-------------|
| `first_person` | Overall score | "I see everything through the lens of..." |
| `logic_of_appropriateness` | Coverage | Third-person describing decision framework |
| `rule_based` | Convex hull | If-then behavioral rules |

**Key finding from evolution:** All formative-memory generators (childhood backstories) were **eliminated**. Action-oriented, present-focused descriptions outperformed memory-grounded ones.

In [12]:
print("=" * 70)
print(f"STEP 2c: STAGE 2 — PARALLEL PERSONA EXPANSION ({PERSONA_FORMAT})")
print("=" * 70)
print(f"\nExpanding {len(stage1_results)} descriptors into full personas...")
print(f"Format: {PERSONA_FORMAT}\n")

personas = []
for i, s1 in enumerate(stage1_results):
    print(f"  Expanding {i+1}/{len(stage1_results)}: {s1['name']}...")

    full_description = expand_persona_stage2(
        name=s1["name"],
        descriptor=s1["descriptor"],
        axis_positions=s1["axis_positions"],
        context=questionnaire.context,
        persona_format=PERSONA_FORMAT,
    )

    personas.append(Persona(
        name=s1["name"],
        stage1_descriptor=s1["descriptor"],
        full_description=full_description,
        axis_positions=s1["axis_positions"],
        persona_format=PERSONA_FORMAT,
    ))

print(f"\n✅ Generated {len(personas)} complete personas")

STEP 2c: STAGE 2 — PARALLEL PERSONA EXPANSION (first_person)

Expanding 25 descriptors into full personas...
Format: first_person

  Expanding 1/25: Yumi...
  Expanding 2/25: Kazuo...
  Expanding 3/25: Rina...
  Expanding 4/25: Taro...
  Expanding 5/25: Aki...
  Expanding 6/25: Sachiko...
  Expanding 7/25: Kaito...
  Expanding 8/25: Yuka...
  Expanding 9/25: Akira...
  Expanding 10/25: Nanako...
  Expanding 11/25: Takashi...
  Expanding 12/25: Erika...
  Expanding 13/25: Satoshi...
  Expanding 14/25: Mika...
  Expanding 15/25: Tetsuya...
  Expanding 16/25: Hiroshi...
  Expanding 17/25: Yoriko...
  Expanding 18/25: Katsumi...
  Expanding 19/25: Reiko...
  Expanding 20/25: Takeshi...
  Expanding 21/25: Naomi...
  Expanding 22/25: Kenji...
  Expanding 23/25: Yoshiko...
  Expanding 24/25: Takao...
  Expanding 25/25: Emiko...

✅ Generated 25 complete personas


In [13]:
# ── Save Stage 2 artifact ──
stage2_artifact = {
    "num_personas": len(personas),
    "persona_format": PERSONA_FORMAT,
    "context": questionnaire.context,
    "dimensions": dimensions,
    "personas": [
        {
            "persona_index": i,
            "name": p.name,
            "target_axis_positions": p.axis_positions,
            "stage1_descriptor": p.stage1_descriptor,
            "stage2_full_description": p.full_description,
            "format": p.persona_format,
            "description_length_chars": len(p.full_description),
        }
        for i, p in enumerate(personas)
    ],
    "statistics": {
        "avg_description_length": np.mean([len(p.full_description) for p in personas]),
        "min_description_length": min(len(p.full_description) for p in personas),
        "max_description_length": max(len(p.full_description) for p in personas),
    },
}

save_artifact(stage2_artifact, "02c_stage2_personas.json")

  ✅ Saved: outputs\02c_stage2_personas.json (71.9 KB)


WindowsPath('outputs/02c_stage2_personas.json')

In [14]:
stage2_artifact = load_artifact("02c_stage2_personas.json")

NUM_PERSONAS = stage2_artifact["num_personas"]
PERSONA_FORMAT = stage2_artifact["persona_format"]
dimensions = stage2_artifact["dimensions"]
context = stage2_artifact["context"]

# Rebuild Persona objects in the original format
personas = [
    Persona(
        name=p_data["name"],
        stage1_descriptor=p_data["stage1_descriptor"],
        full_description=p_data["stage2_full_description"],
        axis_positions=p_data["target_axis_positions"],
        persona_format=p_data["format"],
    )
    for p_data in stage2_artifact["personas"]
]

### 📋 Inspect: Full persona descriptions

Compare the Stage 1 sketch with the Stage 2 expansion. The expansion adds rich detail — occupation, relationships, cognitive lens, decision-making framework — all anchored to the target axis positions.

In [15]:
# Display a few complete personas side-by-side with their Stage 1 input
for i in range(min(3, len(personas))):
    p = personas[i]
    print(f"{'═'*70}")
    print(f"PERSONA {i+1}: {p.name}")
    print(f"{'═'*70}")

    print(f"\nTarget Positions:")
    for dim, info in p.axis_positions.items():
        print(f"  {dim}: {info['value']:.3f} ({info['label']})")

    print(f"\n── Stage 1 (high-level descriptor) ──")
    print(f"{p.stage1_descriptor[:300]}")

    print(f"\n── Stage 2 (full {p.persona_format} description) ──")
    print(f"{p.full_description[:600]}")
    if len(p.full_description) > 600:
        print(f"  ... [{len(p.full_description)} chars total]")
    print()

══════════════════════════════════════════════════════════════════════
PERSONA 1: Yumi
══════════════════════════════════════════════════════════════════════

Target Positions:
  community_cohesion: 0.620 (high)
  technology_adoption: 0.206 (very low)
  adherence_to_tradition: 0.314 (low)

── Stage 1 (high-level descriptor) ──
Yumi is a community-oriented person with strong ties to her neighbors (community_cohesion: 0.62). Despite her strong sense of community, she's hesitant to adopt new technology, such as cell phones or the internet (technology_adoption: 0.21). While she respects tradition, she doesn't prioritize adher

── Stage 2 (full first_person description) ──
I'm Yumi, a 65-year-old wife, mother, and part-time farmer, living in this beautiful rural village in Japan. To me, community is everything - it's the air I breathe, the food I eat, and the people I share my joys and sorrows with. I've always been someone who puts the needs of others before my own, and I take pride in bei

---
## Step 3: Concordia Evaluation — Logic of Appropriateness

**Paper reference:** Section 3.4

Each persona is placed in a simulation where they answer the questionnaire items using the **Logic of Appropriateness** framework (March & Olsen, 2011):

1. **"What kind of situation is this?"** → Interpret the survey question
2. **"What kind of person is [name]?"** → Consult the persona description
3. **"What does a person like [name] do?"** → Select a Likert response

**Critical design choices:**
- **Memory reset** between each question (no carry-over / priming effects)
- Each response is independently scored on the Likert scale
- Scores are **aggregated by mean** per diversity axis → response embedding `z_i ∈ R^K`
- The full population produces `Z ∈ R^{N×K}` — the object we measure diversity over

> This is where the **items `I`** are finally used — the personas have never seen these questions before.

In [17]:
print("=" * 70)
print("STEP 3: CONCORDIA EVALUATION")
print("=" * 70)
print(f"\nEvaluating {len(personas)} personas on {len(questionnaire.questions)} items")
print(f"Dimensions: {dimensions}")
print(f"Memory reset between each question: YES\n")

eval_results = evaluate_population(personas, questionnaire)

print(f"\n✅ Evaluation complete")
print(f"Response embedding shape: {eval_results['embeddings'].shape}")

STEP 3: CONCORDIA EVALUATION

Evaluating 25 personas on 9 items
Dimensions: ['community_cohesion', 'technology_adoption', 'adherence_to_tradition']
Memory reset between each question: YES

  Evaluating persona 1/25: Yumi
  Evaluating persona 2/25: Kazuo
  Evaluating persona 3/25: Rina
  Evaluating persona 4/25: Taro
  Evaluating persona 5/25: Aki
  Evaluating persona 6/25: Sachiko
  Evaluating persona 7/25: Kaito
  Evaluating persona 8/25: Yuka
  Evaluating persona 9/25: Akira
  Evaluating persona 10/25: Nanako
  LLM call attempt 1 failed: API error: {'message': 'Internal Server Error', 'code': 500}. Retrying...
  LLM call attempt 1 failed: API error: {'message': 'Internal Server Error', 'code': 500}. Retrying...
  Evaluating persona 11/25: Takashi
  LLM call attempt 1 failed: API error: {'message': 'Internal Server Error', 'code': 500}. Retrying...
  LLM call attempt 1 failed: API error: {'message': 'Internal Server Error', 'code': 500}. Retrying...
  LLM call attempt 1 failed: API er

In [18]:
# ── Save evaluation artifact ──
eval_artifact = {
    "num_personas": len(personas),
    "num_questions": len(questionnaire.questions),
    "dimensions": dimensions,
    "embedding_shape": list(eval_results["embeddings"].shape),
    "embeddings_Z": eval_results["embeddings"].tolist(),
    "per_persona_scores": eval_results["per_persona_scores"],
    "per_persona_detail": [
        {
            "name": personas[i].name,
            "target_positions": personas[i].axis_positions,
            "measured_scores": eval_results["per_persona_scores"][i],
            "individual_responses": eval_results["raw_responses"][i],
        }
        for i in range(len(personas))
    ],
    "population_statistics": {
        dim: {
            "mean": float(np.mean(eval_results["embeddings"][:, j])),
            "std": float(np.std(eval_results["embeddings"][:, j])),
            "min": float(np.min(eval_results["embeddings"][:, j])),
            "max": float(np.max(eval_results["embeddings"][:, j])),
            "range": float(np.ptp(eval_results["embeddings"][:, j])),
        }
        for j, dim in enumerate(dimensions)
    },
}

save_artifact(eval_artifact, "03_evaluation_results.json")

  ✅ Saved: outputs\03_evaluation_results.json (82.5 KB)


WindowsPath('outputs/03_evaluation_results.json')

### 📋 Inspect: Response Embeddings Z

Each persona is now a **numerical point** in K-dimensional diversity space. We can see:
- Whether the target positions (from Sobol sampling) translated into actual behavioral diversity
- Whether extreme targets produced extreme responses
- The spread and coverage of the population

In [19]:
# Display the response embeddings table
print_evaluation_results(personas, eval_results)


EVALUATION RESULTS: Response Embeddings Z
Persona                community_cohesion  technology_adoption adherence_to_traditi
-----------------------------------------------------------------------------------
Yumi                                 4.33                 1.33                 3.00
Kazuo                                2.67                 1.33                 2.67
Rina                                 2.67                 4.33                 2.67
Taro                                 4.33                 1.33                 3.00
Aki                                  2.33                 4.33                 1.33
Sachiko                              4.33                 1.33                 3.33
Kaito                                4.33                 4.33                 3.67
Yuka                                 4.33                 1.33                 4.33
Akira                                2.67                 4.33                 2.67
Nanako                           

In [20]:
# Compare target positions vs measured scores
print("\nTarget vs Measured Comparison (first 10 personas):\n")
header = f"{'Name':<15}"
for dim in dimensions:
    short = dim[:12]
    header += f" {'Tgt':>5} {'Msd':>5}"
print(header)
print("─" * len(header))

Z = eval_results["embeddings"]
for i in range(min(10, len(personas))):
    p = personas[i]
    row = f"{p.name:<15}"
    for j, dim in enumerate(dimensions):
        target = p.axis_positions[dim]["value"]
        # Map target [0,1] → [1,5] for comparison with Likert scores
        target_likert = target * 4.0 + 1.0
        measured = Z[i, j]
        row += f" {target_likert:>5.2f} {measured:>5.2f}"
    print(row)

print(f"\n(Tgt = target Sobol position mapped to [1,5], Msd = measured Likert mean)")


Target vs Measured Comparison (first 10 personas):

Name              Tgt   Msd   Tgt   Msd   Tgt   Msd
───────────────────────────────────────────────────
Yumi             3.48  4.33  1.82  1.33  2.26  3.00
Kazuo            2.90  2.67  1.46  1.33  2.56  2.67
Rina             2.18  2.67  4.69  4.33  2.82  2.67
Taro             4.07  4.33  1.33  1.33  1.93  3.00
Aki              2.28  2.33  3.36  4.33  1.76  1.33
Sachiko          4.52  4.33  1.64  1.33  3.02  3.33
Kaito            4.30  4.33  4.40  4.33  3.34  3.67
Yuka             4.96  4.33  2.31  1.33  4.08  4.33
Akira            1.83  2.67  4.75  4.33  3.59  2.67
Nanako           2.72  2.67  4.26  4.33  4.22  4.33

(Tgt = target Sobol position mapped to [1,5], Msd = measured Likert mean)


---
## Step 4: Diversity Metrics

**Paper reference:** Section 3.5 and Appendix C

Six metrics quantify how well the population covers the diversity space:

### Maximize ↑
| Metric | What it measures |
|--------|-----------------|
| **Coverage** | Fraction of the space within radius *k* of at least one persona (Monte Carlo estimate) |
| **Convex Hull Volume** | Volume of the smallest convex shape containing all persona points |
| **Mean Pairwise Distance** | Average spread between all pairs of personas |
| **Min Pairwise Distance** | Distance between the two closest personas (no duplicates) |

### Minimize ↓
| Metric | What it measures |
|--------|-----------------|
| **Dispersion** | Radius of the largest empty region (gaps in coverage) |
| **KL Divergence** | How far the empirical distribution is from an ideal quasi-random one |

The paper's evolved generators achieved **>80% coverage** on held-out test sets.

In [21]:
print("=" * 70)
print("STEP 4: DIVERSITY METRICS")
print("=" * 70)
print(f"\nComputing all 6 metrics on Z with shape {eval_results['embeddings'].shape}...")
print(f"(Normalizing from Likert [1,5] to [0,1] before computing)\n")

metrics = compute_all_metrics(
    eval_results["embeddings"],
    seed=SEED,
    fast_mode=False,
)

print_metrics(metrics)

STEP 4: DIVERSITY METRICS

Computing all 6 metrics on Z with shape (25, 3)...
(Normalizing from Likert [1,5] to [0,1] before computing)


  Computing diversity metrics (N=25, K=3)...
    Coverage: 0.9709
    Convex Hull Volume: 0.3576
    Min Pairwise Distance: 0.0000
    Mean Pairwise Distance: 0.6955
    Dispersion: 0.5126
    KL Divergence: 1.1303

DIVERSITY METRICS
  coverage                         0.9709  ↑ higher is better
  convex_hull_volume               0.3576  ↑ higher is better
  min_pairwise_distance            0.0000  ↑ higher is better
  mean_pairwise_distance           0.6955  ↑ higher is better
  dispersion                       0.5126  ↓ lower is better
  kl_divergence                    1.1303  ↓ lower is better


In [22]:
# ── Save diversity metrics artifact ──
Z_norm = normalize_embeddings(eval_results["embeddings"])

metrics_artifact = {
    "population_size": NUM_PERSONAS,
    "num_dimensions": K,
    "dimensions": dimensions,
    "persona_format": PERSONA_FORMAT,
    "metrics": metrics,
    "interpretation": {
        "coverage": {
            "value": metrics["coverage"],
            "direction": "higher is better",
            "paper_target": ">0.80 on held-out test sets",
            "description": "Fraction of diversity space within reach of at least one persona",
        },
        "convex_hull_volume": {
            "value": metrics["convex_hull_volume"],
            "direction": "higher is better",
            "max_possible": 1.0,
            "description": "Volume of the convex hull of all persona embeddings in [0,1]^K",
        },
        "mean_pairwise_distance": {
            "value": metrics["mean_pairwise_distance"],
            "direction": "higher is better",
            "description": "Average Euclidean distance between all pairs of personas",
        },
        "min_pairwise_distance": {
            "value": metrics["min_pairwise_distance"],
            "direction": "higher is better",
            "description": "Distance between the two most similar personas (no duplicates)",
        },
        "dispersion": {
            "value": metrics["dispersion"],
            "direction": "lower is better",
            "description": "Radius of the largest empty ball (biggest gap in coverage)",
        },
        "kl_divergence": {
            "value": metrics["kl_divergence"],
            "direction": "lower is better",
            "description": "KL divergence from an ideal quasi-random distribution",
        },
    },
    "normalized_embeddings": Z_norm.tolist(),
}

save_artifact(metrics_artifact, "04_diversity_metrics.json")

  ✅ Saved: outputs\04_diversity_metrics.json (4.0 KB)


WindowsPath('outputs/04_diversity_metrics.json')

---
## Pipeline Summary

In [23]:
print("=" * 70)
print("PIPELINE COMPLETE — SUMMARY")
print("=" * 70)

print(f"\n📝 Input: \"{SHORT_DESCRIPTION}\"")
print(f"\n📊 Questionnaire:")
print(f"   Context: {questionnaire.context[:100]}...")
print(f"   Axes: {dimensions}")
print(f"   Items: {len(questionnaire.questions)}")

print(f"\n👥 Population: {NUM_PERSONAS} personas ({PERSONA_FORMAT} format)")

print(f"\n📈 Diversity Metrics:")
maximize = ["coverage", "convex_hull_volume", "mean_pairwise_distance", "min_pairwise_distance"]
minimize = ["dispersion", "kl_divergence"]
for name in maximize:
    print(f"   ↑ {name:<30} {metrics[name]:.4f}")
for name in minimize:
    print(f"   ↓ {name:<30} {metrics[name]:.4f}")

print(f"\n📁 Saved Artifacts:")
for f in sorted(OUTPUT_DIR.glob("*.json")):
    size_kb = f.stat().st_size / 1024
    print(f"   {f.name:<40} ({size_kb:.1f} KB)")

PIPELINE COMPLETE — SUMMARY

📝 Input: "elderly rural japan 2010"

📊 Questionnaire:
   Context: A survey of elderly residents in a rural Japanese village in 2010, exploring their feelings about co...
   Axes: ['community_cohesion', 'technology_adoption', 'adherence_to_tradition']
   Items: 9

👥 Population: 25 personas (first_person format)

📈 Diversity Metrics:
   ↑ coverage                       0.9709
   ↑ convex_hull_volume             0.3576
   ↑ mean_pairwise_distance         0.6955
   ↑ min_pairwise_distance          0.0000
   ↓ dispersion                     0.5126
   ↓ kl_divergence                  1.1303

📁 Saved Artifacts:
   01_questionnaire.json                    (4.8 KB)
   02a_diversity_positions.json             (13.1 KB)
   02b_stage1_descriptors.json              (24.2 KB)
   02c_stage2_personas.json                 (71.9 KB)
   03_evaluation_results.json               (82.5 KB)
   04_diversity_metrics.json                (4.0 KB)


---
## Appendix: Inspect Individual Persona Evaluations

Drill into how specific personas answered each question.

In [ ]:
# Pick a few interesting personas to inspect in detail
for idx in [0, len(personas)//2, len(personas)-1]:
    if idx >= len(personas):
        continue
    p = personas[idx]
    responses = eval_results["raw_responses"][idx]
    scores = eval_results["per_persona_scores"][idx]

    print(f"{'═'*70}")
    print(f"DETAILED EVALUATION: {p.name} (Persona {idx+1})")
    print(f"{'═'*70}")
    print(f"\nTarget positions:")
    for dim, info in p.axis_positions.items():
        print(f"  {dim}: {info['value']:.3f} ({info['label']})")
    print(f"\nMeasured scores:")
    for dim, score in scores.items():
        print(f"  {dim}: {score:.2f}")

    print(f"\nIndividual responses:")
    for r in responses:
        rev = " [REV]" if r["reverse_coded"] else ""
        stmt = r["statement"][:60]
        # Extract just the choice from the response
        resp_short = r["response"][:50].replace("\n", " ")
        print(f"  [{r['dimension'][:15]:<15}]{rev:<6} → {r['score']:.0f}/5  |  {resp_short}")
    print()